# WebGL 01 -- Graphics Math: Transforms, Projection & Software Rasterizer

> **Geo-Instructor . WebGL/Three.js Career Track . Notebook 1 of 3**

Every 3D scene goes through the same pipeline: object space -> world -> camera -> clip -> screen.
This notebook builds every stage from scratch in numpy, then rasterizes a triangle by hand.

```
Runtime: ~3 min  |  No GPU  |  numpy + matplotlib only
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
np.random.seed(0)
plt.rcParams.update({'figure.facecolor':'#F4F2F0','axes.facecolor':'#F4F2F0','font.family':'monospace','axes.grid':True,'grid.alpha':0.3})

# Unit cube vertices and edges
cube_verts = np.array([
    [-0.5,-0.5,-0.5],[0.5,-0.5,-0.5],[0.5,0.5,-0.5],[-0.5,0.5,-0.5],
    [-0.5,-0.5, 0.5],[0.5,-0.5, 0.5],[0.5,0.5, 0.5],[-0.5,0.5, 0.5],
], dtype=float)
cube_edges = [(0,1),(1,2),(2,3),(3,0),(4,5),(5,6),(6,7),(7,4),(0,4),(1,5),(2,6),(3,7)]
print('Cube: 8 vertices, 12 edges. Ready.')

---
## Part 1 -- Model Transforms: T * R * S

Every object in a scene has a **model matrix** that brings it from local (object) space to world space.
Convention: apply Scale first, then Rotate, then Translate (right-to-left matrix multiplication).

```
  Model = T * R * S
  world_pos = Model * local_pos
```

In [ ]:
def translate(tx,ty,tz):
    M=np.eye(4); M[0,3]=tx; M[1,3]=ty; M[2,3]=tz; return M
def scale(sx,sy,sz):
    return np.diag([sx,sy,sz,1.0])
def rot_x(a):
    c,s=np.cos(a),np.sin(a)
    M=np.eye(4); M[1,1]=c; M[1,2]=-s; M[2,1]=s; M[2,2]=c; return M
def rot_y(a):
    c,s=np.cos(a),np.sin(a)
    M=np.eye(4); M[0,0]=c; M[0,2]=s; M[2,0]=-s; M[2,2]=c; return M
def rot_z(a):
    c,s=np.cos(a),np.sin(a)
    M=np.eye(4); M[0,0]=c; M[0,1]=-s; M[1,0]=s; M[1,1]=c; return M

def transform_pts(M, pts):
    h = np.hstack([pts, np.ones((len(pts),1))])  # homogeneous
    t = (M @ h.T).T
    return t[:,:3]

# Build model matrix: scale x2, rotate 30deg around Y, translate to (1,0,0)
S = scale(2.0, 1.0, 1.0)
R = rot_y(np.radians(30))
T = translate(1.0, 0.5, 0.0)
model = T @ R @ S

fig = plt.figure(figsize=(16, 4))
stages = [
    ('Local space', cube_verts, 'gray'),
    ('After Scale', transform_pts(S, cube_verts), 'steelblue'),
    ('After Rotate', transform_pts(R@S, cube_verts), 'seagreen'),
    ('After Translate (World)', transform_pts(model, cube_verts), 'tomato'),
]
for i, (title, verts, color) in enumerate(stages):
    ax = fig.add_subplot(1, 4, i+1, projection='3d')
    for a,b in cube_edges:
        ax.plot([verts[a,0],verts[b,0]],[verts[a,1],verts[b,1]],[verts[a,2],verts[b,2]],color=color,lw=1.5)
    ax.set_title(title, fontsize=8); ax.set_xlim(-3,3); ax.set_ylim(-3,3); ax.set_zlim(-3,3)
plt.suptitle('Model matrix: T * R * S applied step by step', fontsize=10)
plt.tight_layout(); plt.show()

---
## Part 2 -- View Matrix: The LookAt Camera

The view matrix is the inverse of the camera's world transform.
Given camera position `eye`, target `center`, and up vector `up`:

```
  forward = normalize(eye - center)
  right   = normalize(cross(up, forward))
  up_true = cross(forward, right)
  View    = rotation * translation  (both derived from above)
```

In [ ]:
def look_at(eye, center, up):
    f = np.array(eye) - np.array(center); f /= np.linalg.norm(f)  # forward
    r = np.cross(np.array(up), f); r /= np.linalg.norm(r)          # right
    u = np.cross(f, r)                                              # true up
    M = np.eye(4)
    M[0,:3]=r; M[1,:3]=u; M[2,:3]=f
    M[0,3]=-np.dot(r,eye); M[1,3]=-np.dot(u,eye); M[2,3]=-np.dot(f,eye)
    return M

eye    = np.array([3.0, 4.0, 5.0])
center = np.array([0.0, 0.0, 0.0])
up     = np.array([0.0, 1.0, 0.0])
V = look_at(eye, center, up)

world_verts = transform_pts(model, cube_verts)
cam_verts   = transform_pts(V, world_verts)

fig = plt.figure(figsize=(13, 5))
for i, (verts, title, color, space) in enumerate([
    (world_verts, 'World space', 'tomato', ((-3,3),(-1,3),(-3,3))),
    (cam_verts, 'Camera space', 'steelblue', ((-3,3),(-3,3),(-8,-1))),
]):
    ax = fig.add_subplot(1, 2, i+1, projection='3d')
    for a,b in cube_edges:
        ax.plot([verts[a,0],verts[b,0]],[verts[a,1],verts[b,1]],[verts[a,2],verts[b,2]],color=color,lw=1.5)
    if i==0:
        ax.scatter(*eye, s=80, c='k', marker='^', zorder=5, label='Camera')
        ax.plot([eye[0],0],[eye[1],0],[eye[2],0],'k--',lw=1,alpha=0.5,label='View dir')
        ax.legend(fontsize=8)
    ax.set_title(title, fontsize=9)
    ax.set_xlabel('X'); ax.set_ylabel('Y'); ax.set_zlabel('Z')
plt.suptitle('LookAt: transforms world into camera space (camera at origin, looking -Z)', fontsize=10)
plt.tight_layout(); plt.show()
print('In camera space: camera is at origin, scene in front is at negative Z.')

---
## Part 3 -- Perspective Projection

```
  fovy  = vertical field of view (radians)
  aspect = width / height
  near, far = clipping planes

  f = 1 / tan(fovy/2)
  Projection matrix (OpenGL convention, NDC z in [-1,1]):
  [ f/aspect   0       0              0       ]
  [   0        f       0              0       ]
  [   0        0   (far+near)/(near-far)  2*far*near/(near-far) ]
  [   0        0      -1              0       ]
```
After projection: divide xyz by w to get NDC in [-1,1]^3.

In [ ]:
def perspective(fovy, aspect, near, far):
    f = 1.0 / np.tan(fovy / 2)
    P = np.zeros((4,4))
    P[0,0] = f/aspect; P[1,1] = f
    P[2,2] = (far+near)/(near-far); P[2,3] = 2*far*near/(near-far)
    P[3,2] = -1.0
    return P

def orthographic(left, right, bottom, top, near, far):
    P = np.zeros((4,4))
    P[0,0] = 2/(right-left); P[0,3] = -(right+left)/(right-left)
    P[1,1] = 2/(top-bottom); P[1,3] = -(top+bottom)/(top-bottom)
    P[2,2] = -2/(far-near);  P[2,3] = -(far+near)/(far-near)
    P[3,3] = 1.0
    return P

def project_to_ndc(P, pts_3d):
    h = np.hstack([pts_3d, np.ones((len(pts_3d),1))])
    clip = (P @ h.T).T
    w = clip[:,3:4]
    ndc = clip[:,:3] / (w + 1e-10)  # perspective divide
    return ndc

P_persp = perspective(np.radians(60), 1.5, 0.1, 100)
P_ortho = orthographic(-2.5, 2.5, -2, 2, 0.1, 100)

ndc_persp = project_to_ndc(P_persp, cam_verts)
ndc_ortho  = project_to_ndc(P_ortho, cam_verts)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, (ndc, title) in zip(axes[:2], [
    (ndc_persp, 'Perspective (FOV 60deg)'),
    (ndc_ortho,  'Orthographic'),
]):
    for a,b in cube_edges:
        ax.plot([ndc[a,0],ndc[b,0]],[ndc[a,1],ndc[b,1]],'steelblue',lw=1.5)
    ax.set_xlim(-1.5,1.5); ax.set_ylim(-1.5,1.5); ax.set_aspect('equal')
    ax.set_title(title); ax.axhline(0,color='k',lw=0.5,alpha=0.3); ax.axvline(0,color='k',lw=0.5,alpha=0.3)
    ax.set_xlabel('NDC X'); ax.set_ylabel('NDC Y')

# FOV comparison
for fov, color, ls in [(30,'tomato','-'), (60,'steelblue','--'), (90,'seagreen',':')]:
    P = perspective(np.radians(fov), 1.5, 0.1, 100)
    ndc = project_to_ndc(P, cam_verts)
    for a,b in cube_edges:
        axes[2].plot([ndc[a,0],ndc[b,0]],[ndc[a,1],ndc[b,1]],color=color,lw=1.2,ls=ls)
from matplotlib.lines import Line2D
axes[2].legend([Line2D([0],[0],color=c) for c in ['tomato','steelblue','seagreen']],
               ['FOV 30','FOV 60','FOV 90'], fontsize=8)
axes[2].set_title('FOV comparison'); axes[2].set_aspect('equal')
axes[2].set_xlim(-2,2); axes[2].set_ylim(-2,2)
plt.suptitle('Perspective vs Orthographic projection -- same camera, same scene', fontsize=10)
plt.tight_layout(); plt.show()
print('Perspective: narrow FOV = telephoto (zoomed), wide FOV = fisheye.')
print('Orthographic: used in CAD/engineering -- no foreshortening.')

---
## Part 4 -- Non-Linear Depth & Z-Fighting

The perspective matrix maps depth non-linearly into NDC Z in [-1,1].
Most precision is near the camera. Near the far plane, many world-space depths
map to the same integer depth buffer value -- causing Z-fighting.

In [ ]:
near, far = 0.5, 100.0
depths_world = np.linspace(near*1.01, far*0.99, 500)
# NDC z: from perspective matrix
def world_to_ndcz(z, near, far):
    return (-(far+near)/(far-near) - 2*far*near/((far-near)*z))

ndc_z = world_to_ndcz(depths_world, near, far)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))
ax1.plot(depths_world, (ndc_z+1)/2, 'steelblue', lw=2)
ax1.set_xlabel('World depth Z'); ax1.set_ylabel('Depth buffer value [0,1]')
ax1.set_title(f'Non-linear depth buffer\nnear={near}, far={far}')
ax1.axvspan(near, near*5, alpha=0.1, color='tomato', label='High precision zone')
ax1.axvspan(far*0.8, far, alpha=0.1, color='gray', label='Z-fighting zone')
ax1.legend(fontsize=8)

# Show effect of near plane distance
for n, color in [(0.1,'seagreen'), (0.5,'steelblue'), (2.0,'tomato'), (10.0,'orange')]:
    nz = world_to_ndcz(depths_world, n, far)
    ax2.plot(depths_world, (nz+1)/2, color=color, lw=1.5, label=f'near={n}')
ax2.set_xlabel('World depth'); ax2.set_ylabel('Depth buffer value')
ax2.set_title('Effect of near plane: closer near = worse distribution'); ax2.legend(fontsize=8)
plt.tight_layout(); plt.show()
print('Rule: set near as far as possible from the camera. This spreads depth precision across the scene.')

In [ ]:
# Software rasterizer: color a triangle using barycentric coordinates
def edge_fn(ax,ay,bx,by,px,py): return (px-ax)*(by-ay) - (py-ay)*(bx-ax)

def rasterize_triangle(v0,v1,v2,c0,c1,c2,W=400,H=300):
    """Rasterize a triangle with per-vertex colors. Returns (H,W,3) image."""
    img = np.ones((H,W,3))
    minx=max(0,int(min(v0[0],v1[0],v2[0])))
    maxx=min(W-1,int(max(v0[0],v1[0],v2[0]))+1)
    miny=max(0,int(min(v0[1],v1[1],v2[1])))
    maxy=min(H-1,int(max(v0[1],v1[1],v2[1]))+1)
    area = edge_fn(v0[0],v0[1],v1[0],v1[1],v2[0],v2[1])
    if abs(area) < 1: return img
    ys,xs = np.mgrid[miny:maxy+1, minx:maxx+1]
    px = xs.astype(float)+0.5; py = ys.astype(float)+0.5
    w0 = edge_fn(v1[0],v1[1],v2[0],v2[1],px,py)
    w1 = edge_fn(v2[0],v2[1],v0[0],v0[1],px,py)
    w2 = edge_fn(v0[0],v0[1],v1[0],v1[1],px,py)
    inside = (w0 >= 0) & (w1 >= 0) & (w2 >= 0)
    lam0 = w0/area; lam1 = w1/area; lam2 = w2/area
    color = lam0[:,:,None]*c0 + lam1[:,:,None]*c1 + lam2[:,:,None]*c2
    img[ys[inside], xs[inside]] = np.clip(color[inside], 0, 1)
    return img

W,H = 500,380
v0 = np.array([250, 40]); v1 = np.array([60, 330]); v2 = np.array([440, 330])
c0 = np.array([1.0,0.2,0.2]); c1 = np.array([0.2,1.0,0.2]); c2 = np.array([0.2,0.2,1.0])
img = rasterize_triangle(v0,v1,v2,c0,c1,c2,W,H)

fig, (ax1,ax2) = plt.subplots(1,2,figsize=(13,5))
ax1.imshow(img, origin='upper'); ax1.set_title('Software rasterized triangle\n(per-vertex color via barycentric interp)'); ax1.axis('off')
# Show barycentric weight fields
ys,xs = np.mgrid[0:H,0:W]
px=xs+0.5; py=ys+0.5
area=edge_fn(v0[0],v0[1],v1[0],v1[1],v2[0],v2[1])
lam0=edge_fn(v1[0],v1[1],v2[0],v2[1],px.astype(float),py.astype(float))/area
lam0_clipped = np.where((lam0>=0)&(lam0<=1),lam0,np.nan)
ax2.imshow(lam0_clipped, cmap='RdBu_r',origin='upper',vmin=0,vmax=1)
ax2.set_title('Barycentric weight lambda_0\n(1 at v0, 0 at opposite edge)')
ax2.plot([v0[0],v1[0],v2[0],v0[0]],[v0[1],v1[1],v2[1],v0[1]],'k-',lw=2)
ax2.axis('off')
plt.tight_layout(); plt.show()
print('Barycentric coordinates: interpolate any vertex attribute (color, UV, normal) smoothly across a triangle.')

---
## Exercises

### Exercise 1 -- Back-face culling
Compute the dot product of the triangle normal with the view direction.
If positive (facing away from camera), skip the triangle. Halves the rasterization work.

### Exercise 2 -- Texture mapping
Add UV coordinates per vertex. Interpolate UV with barycentric coordinates.
Sample a texture image at the interpolated (u,v).

### Exercise 3 -- Phong shading
For each pixel, interpolate the vertex normals barycentrically.
Compute `diffuse = max(0, dot(N, L))` and `specular = max(0, dot(R, V))^shininess`.

---
## Summary: The MVP Pipeline

| Stage | Matrix | Space | Three.js |
|-------|--------|-------|----------|
| Object -> World | Model (T*R*S) | World space | mesh.matrixWorld |
| World -> Camera | View (LookAt) | Camera space | camera.matrixWorldInverse |
| Camera -> Clip | Projection | Clip space | camera.projectionMatrix |
| Clip -> NDC | / w | NDC [-1,1]^3 | (automatic) |
| NDC -> Screen | Viewport | Pixel coords | renderer.setViewport |

In [ ]:
print('Notebook complete.')
print('Key takeaways:')
print('  Model matrix       -> T * R * S, object to world')
print('  LookAt view matrix -> world to camera space')
print('  Perspective proj   -> FOV controls zoom, depth non-linear')
print('  Barycentric coords -> interpolate vertex attributes across triangle')
print()
print('Production: Three.js handles all of this automatically via WebGLRenderer')
print('  THREE.Matrix4, THREE.PerspectiveCamera, THREE.WebGLRenderer')